# GameTheory 16k — Théorie des enchères : le prix mesuré de l'information

**Objectifs** : analyser la vérité, les revenus et la réserve optimale ; comparer deux formats.

**Prérequis** : espérance, intégration et Python. **Durée** : 60 minutes.

**Navigation** : [Version C#](GameTheory-16k-Theorie-des-Encheres-Myerson-CSharp.ipynb) | [<< 16c-Extraction-de-Revenu-DSIC-IR](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb) | [Index GameTheory](README.md) | [17-MultiAgent-RL >>](GameTheory-17-MultiAgent-RL-Python.ipynb)

**La théorie des enchères** (Myerson, Vickrey) : un vendeur met aux enchères un bien unique,
$n$ acquéreurs ont chacun une **valeur privée** qu'ils seuls connaissent. On **mesure** le
prix que l'asymétrie d'information impose au mécanisme — le symétrique exact de 16j
(screening / loyer d'information), du côté du vendeur.

À la famille **16 (Mécanismes)**, à côté de 16j : on calcule, pour des valeurs
$Uniforme[0,1]$, (1) la **vrais-facité** de l'enchère d'ordre 2 (Vickrey), (2) le **revenu
espéré du vendeur**, (3) la **réserve optimale de Myerson** et le **saut de revenu** qu'elle
produit, (4) le **théorème d'équivalence de revenu** (ordre 1 ≈ ordre 2).

Toutes les valeurs sont **calculées** dans le carnet (numpy, CPU seul, seedé), pas énoncées.


## 1. Le modèle

$n$ acquéreurs, chacun avec une **valeur privée** $v_i$ i.i.d. tirée de $Uniforme[0,1]$.
Le bien part au plus haut, sous une règle de paiement. Deux notions portent tout le reste :

| | Formule | Rôle |
|---|---|---|
| Fonction de répartition | $F(v) = v$  | probabilité qu'une valeur soit $\le v$ |
| Densité | $f(v) = 1$  | densité uniforme sur $[0,1]$ |
| **Valeur virtuelle** | $\varphi(v) = v - \dfrac{1-F(v)}{f(v)} = 2v - 1$  | le poids *économique* d'un acquéreur de valeur $v$ |

La **valeur virtuelle** est la clé de Myerson : en dessous de $r^*$ (où $\varphi=0$), un
acquéreur a une valeur virtuelle **négative** — le vendre, c'est créer de la rente d'information
qui coûte plus qu'il ne rapporte.


In [1]:
import numpy as np

# quadrature robuste : numpy 2.x -> trapezoid, numpy 1.x -> trapz (sans avertissement)
_trap = getattr(np, "trapezoid", None) or np.trapz

n = 3  # nombre d'acquéreurs

# Valeurs privées i.i.d. Uniforme[0, 1]
def F(v):  return np.clip(np.asarray(v, dtype=float), 0.0, 1.0)
def f(v):  return np.where((np.asarray(v, float) >= 0) & (np.asarray(v, float) <= 1), 1.0, 0.0)

# Valeur virtuelle de Myerson : phi(v) = v - (1 - F(v)) / f(v)
def phi(v):
    v = np.asarray(v, dtype=float)
    return v - (1.0 - F(v)) / np.where(f(v) > 0, f(v), 1.0)

# Reserve optimale : phi(r*) = 0  ->  r* = 1/2  pour Uniforme[0,1]
r_star = 0.5

print(f"n = {n} acqéreurs, valeurs i.i.d. Uniforme[0, 1]")
print(f"valeur virtuelle  phi(v) = 2v - 1 :")
for v in [0.0, 0.25, 0.5, 0.75, 1.0]:
    print(f"   v = {v:.2f}  ->  phi = {phi(v):.3f}")
print(f"reserve optimale  r* = {r_star}  (phi = 0 : en dessous, valeur virtuelle negative)")

n = 3 acqéreurs, valeurs i.i.d. Uniforme[0, 1]
valeur virtuelle  phi(v) = 2v - 1 :
   v = 0.00  ->  phi = -1.000
   v = 0.25  ->  phi = -0.500
   v = 0.50  ->  phi = 0.000
   v = 0.75  ->  phi = 0.500
   v = 1.00  ->  phi = 1.000
reserve optimale  r* = 0.5  (phi = 0 : en dessous, valeur virtuelle negative)


### Lecture

$\varphi$ croît de $-1$ (à $v=0$) à $+1$ (à $v=1$) et s'annule à $v=1/2$. Un acquéreur
dont la valeur est **sous** $1/2$ a une valeur virtuelle négative : si on le laisse remporter,
il paie moins que la rente d'information qu'il extrait. La réserve optimale **coupe** juste à
$1/2$ — c'est toute la décision de conception.


### Exercice 1 — Valeur virtuelle pour $F(v)=v^2$

Écrivez une fonction de $v>0$ qui calcule la valeur virtuelle ; puis cherchez son zéro dans $[0,1]$. Renseignez aussi la réserve optimale dans `reserve_beta` (Python) ou `reserveBeta` (C#). Le garde vérifie plusieurs valeurs et la racine annoncée.

In [2]:
def virtuelle_beta(v):
    # TODO : calculer v - (1-F(v))/f(v)
    return None

reserve_beta = None  # TODO : réserve optimale de cette distribution
sondes = [0.25, 0.5, 0.8]
reponses = [virtuelle_beta(v) for v in sondes]
if any(x is None for x in reponses) or reserve_beta is None:
    print("Exercice 1 à compléter")
else:
    assert all(np.isfinite(x) and abs(x-attendu) < 1e-9 for x,attendu in zip(reponses,[-1.625,-0.25,0.575]))
    assert np.isfinite(reserve_beta) and 0 < reserve_beta < 1
    # Résidu indépendant du callback : F(r)=r² et f(r)=2r.
    residu_distribution = reserve_beta - (1-reserve_beta**2)/(2*reserve_beta)
    assert abs(residu_distribution) < 1e-9
    racine = virtuelle_beta(reserve_beta)
    assert racine is not None and np.isfinite(racine) and abs(racine) < 1e-9
    print("Exercice 1 : valeur virtuelle et réserve validées")

Exercice 1 à compléter


## 2. Vrais-facité de l'enchère d'ordre 2 (Vickrey)

En **enchère d'ordre 2** (au second prix), le gagnant paie la **plus haute offre des autres**
(dite $x$), pas la sienne. Soit $v$ la valeur qu'un acquéreur a réellement pour le bien, et $b$
son **offre**. Il remporte le bien **si et seulement si** $b \ge x$ ; s'il gagne, il paie $x$.
Son surplus est donc :

$$U(v; b, x) = \begin{cases} v - x & \text{si } b \ge x \\ 0 & \text{si } b < x . \end{cases}$$

On **mesure** que révéler sa vraie valeur ($b = v$) maximise ce surplus, **quelle que soit** la
compétition $x$ : l'offre franche est une **stratégie dominante**.


In [3]:
def surplus(v, x, b):
    return (v - x) if b >= x else 0.0

print("L'offre de sa VRAIE valeur v est-elle optimale ?  (ordre 2 / Vickrey)")
print(f"{'v':>5} | {'x (competition)':>15} | {'surplus si b=v':>16} | {'max de surplus':>16} | {'b=v optimal ?':>14}")
print("-" * 74)
for v, x in [(0.9, 0.4), (0.9, 0.8), (0.4, 0.3), (0.4, 0.6), (0.6, 0.6)]:
    b_grid = np.linspace(0, 1, 2001)
    surp_truthful = surplus(v, x, v)
    max_surp = max(surplus(v, x, b) for b in b_grid)
    optimal = "OUI" if abs(surp_truthful - max_surp) < 1e-9 else "non"
    print(f"{v:>5.2f} | {x:>15.2f} | {surp_truthful:>16.3f} | {max_surp:>16.3f} | {optimal:>14}")
print()
print("La revelation franche est une STRATEGIE DOMINANTE : optimale quelle que soit la competition x.")

L'offre de sa VRAIE valeur v est-elle optimale ?  (ordre 2 / Vickrey)
    v | x (competition) |   surplus si b=v |   max de surplus |  b=v optimal ?
--------------------------------------------------------------------------
 0.90 |            0.40 |            0.500 |            0.500 |            OUI
 0.90 |            0.80 |            0.100 |            0.100 |            OUI
 0.40 |            0.30 |            0.100 |            0.100 |            OUI
 0.40 |            0.60 |            0.000 |            0.000 |            OUI
 0.60 |            0.60 |            0.000 |            0.000 |            OUI

La revelation franche est une STRATEGIE DOMINANTE : optimale quelle que soit la competition x.


### Lecture

Sur chaque ligne, offrir sa vraie valeur $v$ atteint le surplus maximal : si $v > x$ elle gagne
et capte $v-x$ ; si $v < x$ elle perd (surplus $0$), ce qui est de toute façon le meilleur
possible. L'acquéreur **n'a pas besoin de modéliser ses adversaires** pour bien jouer — c'est
la propriété qui rend l'ordre 2 si précieux, et que 16j ne pouvait pas offrir.


## 3. Revenu du vendeur — ordre 2 sans réserve

Quand tous offrent franchement, le vendeur d'un ordre 2 **sans réserve** reçoit la **2ᵉ plus
haute valeur** des $n$ acquéreurs. C'est la **2ᵉ statistique d'ordre** $X_{(n-1)}$. Pour des
valeurs $Uniforme[0,1]$, son espérance a une forme close :

$$\mathbb{E}[X_{(n-1)}] = \frac{n-1}{n+1} .$$

On la **mesure** — formule close d'un côté, Monte-Carlo seedé de l'autre.


In [4]:
def rev_2price_noreserve_analytic(n):
    return (n - 1) / (n + 1)

def rev_2price_noreserve_mc(n, T=200000, seed=42):
    rg = np.random.default_rng(seed)
    vals = rg.random((T, n))
    # 2e plus haute valeur = index -2 apres tri croissant
    return vals[np.arange(T), np.argsort(vals, axis=1)[:, -2]].mean()

print("Revenu du vendeur en ordre 2 SANS reserve = E[2e statistique d'ordre]")
print(f"{'n':>4} | {'analytique (n-1)/(n+1)':>24} | {'Monte-Carlo (seed 42)':>22}")
print("-" * 54)
for n in [2, 3, 5, 10]:
    print(f"{n:>4} | {rev_2price_noreserve_analytic(n):>24.4f} | {rev_2price_noreserve_mc(n):>22.4f}")

Revenu du vendeur en ordre 2 SANS reserve = E[2e statistique d'ordre]
   n |   analytique (n-1)/(n+1) |  Monte-Carlo (seed 42)
------------------------------------------------------
   2 |                   0.3333 |                 0.3330
   3 |                   0.5000 |                 0.4995
   5 |                   0.6667 |                 0.6668
  10 |                   0.8182 |                 0.8182


### Lecture

La formule close et le Monte-Carlo diffèrent au plus de $0.0005$ dans les valeurs affichées : le revenu de référence du
vendeur est **mesuré**, pas supposé. Pour $n=3$ il vaut $0.5000$ — c'est le **benchmark** que
la conception de mécanismes va chercher à dépasser.


## 4. La réserve de Myerson — le saut de revenu

Le **théorème de Myerson** : l'enchère qui maximise le revenu espéré du vendeur est un
**ordre 2 avec réserve** $r^*$, où $r^*$ est la valeur de **valeur virtuelle nulle**
($r^*=1/2$ ici). Règle : on vend **si** le plus haut bid atteint $r^*$, et le gagnant paie
$\max(r^*, \text{2ᵉ bid})$.

On **mesure** son revenu espéré — par intégrale (quadrature du triangle des deux plus hautes
valeurs) **et** par Monte-Carlo — et on le compare au benchmark sans réserve.


In [5]:
def rev_optimal_analytic(n, r):
    # E[ max(r, x) * 1{y >= r} ], x = 2e valeur, y = 1re, 0 <= x <= y <= 1.
    # Densite conjointe f(x,y) = n(n-1) x^(n-2). Integre en y d'abord :
    #   x >= r : max=r? non, max(r,x)=x, et y>=x>=r -> facteur (1-x)
    #   x <  r : max(r,x)=r, et y doit >= r     -> facteur (1-r)
    xs = np.linspace(0, 1, 20001)
    term = np.where(xs >= r, xs * (1 - xs), r * (1 - r))
    return _trap(term * n * (n - 1) * xs**(n - 2), xs)

def rev_optimal_mc(n, r, T=400000, seed=42):
    rg = np.random.default_rng(seed)
    v = rg.random((T, n))
    top2 = np.sort(v, axis=1)[:, -2:]            # [2e, 1re]
    payment = np.where(top2[:, 1] >= r, np.maximum(r, top2[:, 0]), 0.0)
    return payment.mean()

n = 3
base = rev_2price_noreserve_analytic(n)
opt  = rev_optimal_analytic(n, r_star)
opt_mc = rev_optimal_mc(n, r_star)
lift = opt - base
print(f"n = {n} acqéreurs, Uniforme[0,1], reserve r* = {r_star} :")
print(f"  ordre 2 SANS reserve (benchmark)             : {base:.5f}")
print(f"  ordre 2 AVEC reserve (optimal de Myerson)    : analytique = {opt:.5f}   Monte-Carlo = {opt_mc:.5f}")
print(f"  SAUT DE REVENU de la reserve                 : {lift:.5f}   (+{100*lift/base:.2f} %)")
assert abs(opt - opt_mc) < 5e-3, "integrale et Monte-Carlo divergent"


n = 3 acqéreurs, Uniforme[0,1], reserve r* = 0.5 :
  ordre 2 SANS reserve (benchmark)             : 0.50000
  ordre 2 AVEC reserve (optimal de Myerson)    : analytique = 0.53125   Monte-Carlo = 0.53076
  SAUT DE REVENU de la reserve                 : 0.03125   (+6.25 %)


### Lecture — le résultat central

Pour $n=3$, la réserve $1/2$ fait monter le revenu espéré de $0.50000$ à $0.53125$
(**$+6.25\%$**), et l'intégrale et le Monte-Carlo s'accordent. C'est le **prix mesuré de
l'information** du côté vendeur : la réserve réduit la rente attendue de certains acquéreurs, au prix de ventes abandonnées. Modeste pour $n=3$ — et c'est
là que le §6 révèle l'essentiel.


### Mais « 0.53125 » est une fraction — la quadrature du §4 l'approchait

La fonction `rev_optimal_analytic` du §4 porte un nom d'« analytique », mais elle
**numériquise** : c'est une quadrature au trapèze sur 20 001 points, pas une forme
close. Sur l'instance exacte du carnet — $n=3$, valeurs $Uniforme[0,1]$, réserve
$r^*=1/2$ — l'intégrale du revenu **ferme** en deux intégrales de polynômes, et le
résultat est une **fraction rationnelle**, calculable à la main. On la calcule ici
en arithmétique **exacte** (`fractions.Fraction`), sans flottant ni quadrature.


In [6]:
from fractions import Fraction

def revenu_optimal_exact(n, r):
    """E[ max(r, x) * 1{y >= r} ] en arithmetique EXACTE (aucune quadrature).
    x = 2e valeur, y = 1re, densite conjointe f(x,y) = n(n-1) x^(n-2) :
      x >= r : max(r,x) = x,  y dans [x,1]      ->  x (1-x)
      x <  r : max(r,x) = r,  y dans [r,1]      ->  r (1-r)
    """
    n, r = Fraction(n), Fraction(r)
    A = r * (1 - r) * (r ** (n - 1)) / (n - 1)          # x <  r
    def antider(x):                                     # primitive de x^(n-1)(1-x)
        return x ** n / n - x ** (n + 1) / (n + 1)
    B = antider(Fraction(1)) - antider(r)               # x >= r
    return n * (n - 1) * (A + B)

n = 3
r_exact = Fraction(1, 2)      # r* = 1/2 (valeur virtuelle nulle), EXACT
base = Fraction(n - 1, n + 1) # (n-1)/(n+1) = 1/2, EXACT
opt = revenu_optimal_exact(n, r_exact)
saut = opt - base
pct = saut / base

print("FORME EXACTE du 'resultat central' (fractions, sans quadrature) :")
print(f"  ordre 2 SANS reserve (benchmark)        = {base}  = {float(base):.5f}")
print(f"  ordre 2 AVEC reserve r* = 1/2 (Myerson) = {opt}  = {float(opt):.5f}")
print(f"  SAUT DE REVENU de la reserve            = {saut}  = {float(saut):.5f}")
print(f"  en proportion du benchmark              = {pct}  = {100*float(pct):.4f} %")

# Recolle a la quadrature 20001 points du §4 (cellule 'rev_optimal_analytic') :
quad = rev_optimal_analytic(n, r_star)
print(f"\n  controle : quadrature du §4 = {quad:.5f}  vs  exact {float(opt):.5f}  "
      f"(ecart {abs(quad - float(opt)):.2e})")

assert opt == Fraction(17, 32), f"attendu 17/32, obtenu {opt}"
assert base == Fraction(1, 2)
assert saut == Fraction(1, 32)
assert pct == Fraction(1, 16)
assert abs(quad - float(opt)) < 1e-4, "exact et quadrature divergent"
print("\n  -> le 0.53125 du §4 est la fraction 17/32 ; le saut est 1/32 (+6.25 % = 1/16).")


FORME EXACTE du 'resultat central' (fractions, sans quadrature) :
  ordre 2 SANS reserve (benchmark)        = 1/2  = 0.50000
  ordre 2 AVEC reserve r* = 1/2 (Myerson) = 17/32  = 0.53125
  SAUT DE REVENU de la reserve            = 1/32  = 0.03125
  en proportion du benchmark              = 1/16  = 6.2500 %

  controle : quadrature du §4 = 0.53125  vs  exact 0.53125  (ecart 1.56e-09)

  -> le 0.53125 du §4 est la fraction 17/32 ; le saut est 1/32 (+6.25 % = 1/16).


### Lecture — la forme exacte du « résultat central »

Le `0.53125` du §4 **est** la fraction $\tfrac{17}{32}$ ; le « saut de revenu » de
la réserve est exactement $\tfrac{1}{32}$, soit $+6.25\,\%$ ($= \tfrac{1}{16}$) du
benchmark. La quadrature à 20 001 points approchait donc, sans le dire, une fraction
propre : le **prix mesuré de l'information** du vendeur, pour $n=3$, a une forme
close $\tfrac{17}{32}$, vérifiable au crayon.


### Exercice 2 — Paiement avec réserve

Écrivez une fonction retournant l’indice du gagnant (à partir de zéro) et le revenu du vendeur ; sans vente, utilisez -1 pour l’indice pour une liste de trois offres et une réserve. Essayez aussi les cas sans vente et où la réserve détermine le prix.

In [7]:
def paiement_enchere(offres, reserve):
    # TODO : retourner (indice gagnant, revenu), ou (-1, 0) sans vente
    return None

cas = [([0.3,0.8,0.55],0.5,(1,0.55)), ([0.1,0.2,0.3],0.5,(-1,0)), ([0.1,0.4,0.8],0.5,(2,0.5))]
reponses = [paiement_enchere(o,r) for o,r,_ in cas]
if any(x is None for x in reponses):
    print("Exercice 2 à compléter")
else:
    assert all(isinstance(x,tuple) and len(x)==2 and x[0]==attendu[0] and np.isfinite(x[1]) and abs(x[1]-attendu[1])<1e-9 for x,(_,_,attendu) in zip(reponses,cas))
    print("Exercice 2 : sondes validées")

Exercice 2 à compléter


## 5. Équivalence de revenu — ordre 1 ≈ ordre 2

En **enchère d'ordre 1** (au premier prix, le gagnant paie **son** bid), l'équilibre
symétrique est $b(v) = \frac{n-1}{n}\,v$. Le théorème d'équivalence de revenu dit que,
**en espérance pour le vendeur**, l'ordre 1 et l'ordre 2 (sans réserve) rapportent la même
chose. On le **mesure** : on simule l'ordre 1 à l'équilibre et on compare.


In [8]:
def rev_1price_mc(n, T=300000, seed=42):
    rg = np.random.default_rng(seed)
    v = rg.random((T, n))
    bids = (n - 1) / n * v                        # equilibre symetrique
    return bids[np.arange(T), np.argmax(bids, axis=1)].mean()

n = 3
r1 = rev_1price_mc(n)
r2 = rev_2price_noreserve_analytic(n)
print(f"n = {n} :  ordre 1 a l'equilibre (MC) = {r1:.4f}   |   ordre 2 sans reserve (analytique) = {r2:.4f}")
print(f"ecart = {abs(r1 - r2):.4f}   ->   THEOREME D'EQUIVALENCE DE REVENU (a la precision du Monte-Carlo)")

n = 3 :  ordre 1 a l'equilibre (MC) = 0.4999   |   ordre 2 sans reserve (analytique) = 0.5000
ecart = 0.0001   ->   THEOREME D'EQUIVALENCE DE REVENU (a la precision du Monte-Carlo)


### Lecture

L'ordre 1 (où chacun **sous-enchérit** pour ne pas payer trop) et l'ordre 2 (où chacun offre
franchement) drainent le **même** revenu espéré du vendeur. La *forme* de l'enchère change le
comportement, pas le revenu — l'unique bémol étant la réserve, qui est le levier de conception
vraiment efficace (c'est elle qui créait le saut du §4).


## 6. Sensibilité — le saut de réserve en fonction de $n$

La réserve rapporte-t-elle toujours la même chose ? On **mesure** le saut de revenu qu'elle
produit quand $n$ croît.


In [9]:
print(f"{'n':>4} | {'ordre 2 sans reserve':>20} | {'optimal (reserve 0.5)':>22} | {'saut':>9} | {'saut %':>9}")
print("-" * 72)
for n in [2, 3, 5, 8, 12]:
    base = rev_2price_noreserve_analytic(n)
    opt  = rev_optimal_analytic(n, r_star)
    lift = opt - base
    print(f"{n:>4} | {base:>20.4f} | {opt:>22.4f} | {lift:>9.4f} | {100*lift/base:>8.2f} %")
print()
print("La reserve rapporte d'autant plus que la competition est MAIGRE (n petit) :")
print("beaucoup d'acheteurs -> l'ordre 2 sans reserve extrait deja quasi tout le surplus.")

   n | ordre 2 sans reserve |  optimal (reserve 0.5) |      saut |    saut %
------------------------------------------------------------------------
   2 |               0.3333 |                 0.4167 |    0.0833 |    25.00 %
   3 |               0.5000 |                 0.5312 |    0.0312 |     6.25 %
   5 |               0.6667 |                 0.6719 |    0.0052 |     0.78 %
   8 |               0.7778 |                 0.7782 |    0.0004 |     0.06 %
  12 |               0.8462 |                 0.8462 |    0.0000 |     0.00 %

La reserve rapporte d'autant plus que la competition est MAIGRE (n petit) :
beaucoup d'acheteurs -> l'ordre 2 sans reserve extrait deja quasi tout le surplus.


### Lecture

Le saut de la réserve **fond** quand la compétition s'épaissit : $+25\%$ pour $n=2$,
$+6.25\%$ pour $n=3$, et presque rien pour $n=12$. Avec beaucoup d'acheteurs, la 2ᵉ plus
haute enchère monte déjà très haut et capture la rente ; la réserve n'a presque plus de marge.
**Concevoir une réserve est un levier quand le marché est mince, pas quand il est saturé.**


### Exercice 3 — Seuil de compétition

Écrivez une fonction qui trouve le plus petit $n\ge2$ dont le saut relatif de réserve est strictement inférieur au seuil donné. Explorez notamment 1 %.

In [10]:
def seuil_competition(seuil):
    # TODO : parcourir les nombres d'acquéreurs
    return None

seuils = [0.1, 0.01, 0.001]
reponses = [seuil_competition(s) for s in seuils]
if any(x is None for x in reponses):
    print("Exercice 3 à compléter")
else:
    for s, n in zip(seuils,reponses):
        assert isinstance(n, (int, np.integer)) and 2 <= n <= 30
        def saut(k):
            base = rev_2price_noreserve_analytic(k)
            return (rev_optimal_analytic(k, 0.5)-base)/base
        assert saut(n)<s and all(saut(k)>=s for k in range(2,n))
    print("Exercice 3 : sondes validées")

Exercice 3 à compléter


## 7. Exercices — récapitulatif

Les trois exercices répartis ci-dessus portent sur la valeur virtuelle, le paiement et le seuil de compétition. Comparez vos résultats aux lectures des sections 1, 4 et 6.

## À retenir

- **L'ordre 2 est vrai par construction** : révéler sa valeur est une stratégie dominante,
  mesurée ligne par ligne — l'acquéreur n'a pas à deviner ses rivaux.
- **L'asymétrie a un prix du côté vendeur** : la réserve de Myerson ($r^*=1/2$ ici) enlève la
  rente d'information et fait monter le revenu espéré — **mesuré** ici à $+6.25\%$ pour $n=3$.
- **Ce levier s'éteint avec la compétition** : le saut de la réserve fond quand $n$ croît
  ($+25\%$ à $n=2$, quasi nul à $n=12$) — la conception de la réserve est un art de
  **marché mince**.
- **La forme ne change pas le revenu** : ordre 1 et ordre 2 (sans réserve) sont équivalents en
  revenu espéré ; une réserve optimale peut augmenter ce revenu.

C'est le miroir vendeur de 16j : là où 16j mesurait le **loyer d'information** que le
principal paie à l'agent informé, 16k mesure la **réserve** que le vendeur impose pour ne pas
payer cette rente — même théorème (Myerson), deux faces du même mécanisme.

**Trois mesures à retenir.**

- **La vrais-facité de l'ordre 2 est mesurée ligne par ligne** : révéler sa valeur est une stratégie dominante, et l'acquéreur n'a pas à deviner ses rivaux.
- **Le saut de réserve** : la réserve de Myerson `r* = 1/2` fait monter le revenu espéré de `+25.00 %` pour `n = 2`, de `+6.25 %` pour `n = 3`, de `+0.78 %` pour `n = 5`, de `+0.06 %` pour `n = 8`, et de `+0.00 %` pour `n = 12`.
- **L'équivalence de revenu** : à réserve nulle, l'ordre 1 et l'ordre 2 ont le même revenu espéré (`0.3333` pour `n = 2`, `0.5000` pour `n = 3`, `0.6667` pour `n = 5`, `0.7778` pour `n = 8`) ; une réserve optimale peut augmenter ce revenu.

**Réserve de portée.** Les valeurs sont calculées pour des valeurs **Uniforme[0, 1]** et une offre symétrique : le `r* = 1/2` mesuré ici est celui de cette distribution, pas une constante. L'exercice 1 (distribution `F(v) = v²`) déplace la réserve optimale — c'est le sens du titre du carnet, « le prix mesuré de l'information ».


## Vérifiez votre compréhension

1. L'ordre 2 (Vickrey) est vrai par construction : l'acquéreur paie le prix de la seconde meilleure offre. Le vendeur y laisse-t-il quelque chose ?

   <details><summary>Réponse</summary>

   Oui : sans réserve, le vendeur renonce au gain de revenu que peut apporter une réserve optimale. À réserve nulle, l'ordre 1 et l'ordre 2 ont le **même** revenu espéré (`0.3333` pour `n = 2`, `0.5000` pour `n = 3`, `0.6667` pour `n = 5`, `0.7778` pour `n = 8`, `0.8462` pour `n = 12`). C'est le théorème d'équivalence de revenu, vérifié numériquement. Dans ce modèle symétrique les deux formats ont le même revenu et la même allocation en espérance ; leurs offres et paiements réalisés diffèrent. L'ordre 2 ne demande pas de connaître les stratégies adverses, contrairement au calcul de l'équilibre de premier prix.

   </details>

2. Le saut de réserve tombe à `+0.00 %` pour `n = 12`. La réserve ne sert donc plus à rien ?

   <details><summary>Réponse</summary>

   Le gain reste positif pour tout $n$ fini mais devient très petit : $+0.00\%$ est un arrondi. Quand `n` acquéreurs sont en lice, l'ordre 2 sans réserve extrait déjà presque tout le surplus : la rente d'information résiduelle est négligeable, et le seuil de compétition (exercice 3) est précisément l'endroit où le saut passe sous 1 %. La portée du résultat est donc conditionnelle : la réserve est un levier de **marché mince**, et son intérêt disparaît quand la concurrence s'épaissit.

   </details>

3. Le revenu espéré croît avec `n` (`0.3333` pour `n = 2` contre `0.8462` pour `n = 12`). Pourquoi le saut de réserve fond-il en même temps ?

   <details><summary>Réponse</summary>

   Les deux mouvements sont de même sens mais de nature différente. Avec `n` acquéreurs, la seconde meilleure offre se rapproche du maximum (ordre des statistiques) : le vendeur capte une part croissante du surplus, d'où la croissance du revenu. Mais la rente qu'il laisse subsister — la différence entre la valeur du bien et ce que paie le gagnant — devient une part **décroissante** de ce surplus total : la réserve a de moins en moins de matière à retirer. D'où un saut qui s'éteint alors même que le revenu, lui, continue de monter.

   </details>

4. La réserve de Myerson vaut `1/2` ici. Est-ce une constante ?

   <details><summary>Réponse</summary>

   Non : c'est la réserve optimale **pour la distribution posée au §1** (valeurs Uniforme[0, 1], offre symétrique). Le procédé de Myerson ne donne pas un nombre, il donne une règle : la réserve est le point où la **valeur virtuelle** s'annule, et cette valeur virtuelle dépend de la distribution des valeurs. L'exercice 1 (distribution `F(v) = v²`) déplace donc la réserve optimale — et c'est précisément le prix « mesuré » de l'information, non une constante à recopier.

   </details>

## Pour aller plus loin

**Carnet suivant.** [GameTheory-17 — MultiAgent RL](GameTheory-17-MultiAgent-RL-Python.ipynb) -- la suite de la famille : là où ce carnet mesure le prix d'un mécanisme d'enchère statique, le 17 pose la question de l'**apprentissage** des agents dans un jeu répété.

**Lectures.**

- Les références complètes du carnet voisin `GameTheory-16c` listent **Myerson et Satterthwaite (1983)** (*Efficient mechanisms for bilateral trade*) et **Rochet (1987)** (rationalisabilité en contexte quasi-linéaire) — l'appareil théorique du prix de réserve.
- Le mécanisme d'enchère de référence est celui du carnet owner : l'enchère de second prix de Vickrey, dont l'équivalence de revenu avec l'enchère de premier prix est l'un des résultats établis de la famille.

**Carnets voisins.**

- [GameTheory-16j — Screening et loyer d'information](GameTheory-16j-Screening-Loyer-Information-Python.ipynb) -- le miroir : le loyer d'information côté principal, mesuré par le même type de décomposition.
- [GameTheory-16c — Extraction de revenu](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb) -- le designer qui maximise le revenu, et l'écart mesuré entre l'espace qu'il visite et l'espace complet.
- [GameTheory-16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb) -- le carnet owner : enchères de Vickrey et mécanisme VCG, dont l'équivalence de revenu est une conséquence.
